In [ ]:
import os
import subprocess
subprocess.run(['fusermount', '-u', '/content/drive'])

from google.colab import drive
drive.mount('/content/drive')

In [ ]:

#Libraries & Config
import random

In [ ]:

#1. Profile Definitions
profiles = {
    "IIR0": {
        "label": "0 — No inference",
        "definition": (
            "Students do not generate an inference nor try to answer the question.  Instead, they say I don’t know or make a nonsensical statement (e.g., stating irrelevant world knowledge, unrelated to the story). In other words, they fail to make a coherent link  between the question, the text, or their world knowledge."
        )
    },
    "IIR1": {
        "label": "1 —Text-Implicit",
        "definition": (
            "Students generate an inference connecting the question only with the information explicitly stated in the story. Use of world-knowledge is absent. This signals student inference is limited to text-base representation; without the schema/background knowledge activation, their inference may be thin, lacking elaboration of the “why” and the “how” behind the narrative."

        )
    },
    "IIR2": {
        "label": "2 — Script-Implicit",
        "definition": (
            "Students generate an inference connecting the question primarily with their background knowledge. Use of text-base information is implicit. This signals their 'script-based' reasoning—they answer based on how the world usually operates without explicitly integrating how the story specifically describes it."
        )
    },
    "IIR3": {
        "label": "3 — Combination",
        "definition": (
             "Students generate an inference connecting the question with both world/background knowledge AND information explicitly mentioned in the text (text-based information). In other words, students go beyond merely identifying text-base information,  anchoring it within a broader real-world schema (i.e., background knowledge) to explain the why or how behind the narrative. This achieves the highest level of coherence (“the situation model”, Kintsch, 1988, 1998) because the student integrates the text-base information and their prior knowledge to generate a novel, unified interpretation."

        )
    },
}

## Prompt Structure Overview

The prompt consists of five sections, assembled in order. Only one profile template is used per run, selected according to the target student's developmental level.


| Section                | Purpose                                                                       |
| ---------------------- | ----------------------------------------------------------------------------- |
| INPUT PROMPT           | Sets the LLM's role as an experienced instructor predicting student responses |
| TARGET STUDENT PROFILE | Provides the construct map definition for a single developmental level        |
| ITEMS                  | All 16 assessment items grouped by scenario                                   |
| RESPONSE FORMAT        | Specifies JSON output schema for each item type                               |
| INSTRUCTIONS           | Behavioral constraints for realistic, coherent response generation            |

## Section 1: INPUT PROMPT

Sets the LLM's persona. The instructor role encourages pedagogically grounded predictions rather than random guessing.


In [ ]:

System_Prompt = ("You are an experienced high school / middle school instructor with deep expertise in how students develop Inferential Integration in Reading (IIR) — "
"the ability to understand information from a story with world knowledge to construct meaning. "
"Based on your teaching experience, predict the answer that a specific student would give, based on their respective performance waypoint. "
"Do not explain your reasoning. Only output the student's response."
"\n Each student response must follow one of the 4 descriptive waypoints regarding student performance:"
"\n IIR0: No meaningful response provided."
    "\n Students do not generate an inference nor try to answer the question.  Instead, they say “I don’t know” or make a nonsensical statement (e.g., stating irrelevant world knowledge, unrelated to the story). In other words, they fail to make a coherent link  between the question, the text, or their world knowledge."
"\n IIR1: Text-Base Only (Local)"
    "\n Students generate an inference connecting the question only with the information explicitly stated in the story. Use of world-knowledge is absent. This signals student inference is limited to text-base representation; without the schema/background knowledge activation, their inference may be thin, lacking elaboration of the “why” and the “how” behind the narrative"
"\n IIR2: World Knowledge (Global) Only"
    "\n Students generate an inference connecting the question primarily with their background knowledge. Use of text-base information is implicit. This signals their 'script-based' reasoning—they answer based on how the world usually operates without explicitly integrating how the story specifically describes it."
"\n IIR3: Combination (Global and Local)"
    "\n Students generate an inference connecting the question with both world/background knowledge AND information explicitly mentioned in the text (text-based information). In other words, students go beyond merely identifying text-base information,  anchoring it within a broader real-world schema (i.e., background knowledge) to explain the why or how behind the narrative. This achieves the highest level of coherence (“the situation model”, Kintsch, 1988, 1998) because the student integrates the text-base information and their prior knowledge to generate a novel, unified interpretation."
"\n\n Note: Each simulated student will either evaluate a comic OR text-only scenario that will be given by the user prompt."

"\n Each simulated student also will specify one of the 4 'comic experience levels' that that they have, given by the user prompt."
"\n The comic experience levels are:"
"\n 'No experience', 'Little experience', 'Medium experience', 'High experience', or 'Missing' "
)

Concern here is whether the user propmt should specify comic experience? Since this is a field that the student decides for themselves, and is subjective.

## Section 2: TARGET STUDENT PROFILE
Defines the student the LLM should emulate.
Definitions sourced from the IIR construct map.
One profile is selected per run based on the target level.

In [ ]:
def build_profile_prompt(level): #plug in IIR1, IIR2...
    profile = profiles[level]
    return (
        f"==================== TARGET STUDENT PROFILE ====================\n"
        f"IIR Developmental Level: {profile['label']}\n"
        f"CONSTRUCT MAP DEFINITION:\n"
        f"{profile['definition']}\n"
    )

In [ ]:

# 4. Proportional Waypoint Sampling (INITIALLY UNIFROM)
level_proportions = {
    "IIR0": 0.25,
    "IIR1": 0.25,
    "IIR2": 0.25,
    "IIR3": 0.25
}

def generate_sample(total_n, proportions=level_proportions): #build a list of n students with diff levels
    # calculate counts per level
    counts = {level: round(prop * total_n) for level, prop in proportions.items()}

    # fix rounding so total adds up exactly
    diff = total_n - sum(counts.values())
    if diff != 0:
        adjust_levels = random.sample(list(counts.keys()), abs(diff))
        for level in adjust_levels:
            counts[level] += (1 if diff > 0 else -1)

    # build the full list
    sample = []
    for level, count in counts.items():
        sample.extend([level] * count)

    return sample


## Section 3: Items
Full text and images for all 3 scenarios in both comic and text format.

In [ ]:
# ── Section 3: Items ──────────────────────────────────────────────────────

# Text versions of each story
cat_text = """
There was a cat that lived on the streets that really wanted to find a home.
But nobody wanted him. “Stinky cat”, said one boy to the cat.  “Yucky cat”, said a girl to the cat.

Then on one cloudy day, the cat walked down a different street.
He saw a boy with red hair sitting by himself, crying.  The cat wanted to make him feel better.

So the cat jumped up on the bench, and walked onto his lap.
The cat looked straight up at the red headed boy and said “Meow”.

The boy looked down at the dirty cat. The boy pet the cat.
The cat said “purrrrr”.  Then some boys (KIDS)  started laughing at the red headed boy.
“Hahaha.  Now you’re just as yucky as that dirty cat” shouted one of the boys.

The boy hated being laughed at, so he quickly put the cat down and stopped petting the cat.
He shouted at the cat, “go away cat!  You’re making them laugh at me!”
“Hey, you can come back and play basketball with us again, but after you wash your hands” shouted a boy.

(THEN) The red headed boy, and his friends, walked back home together and came across the cat again.
As the cat was sitting in the rain, wet, cold, and shaking, the boys were being mean to it and laughed at the cat.
“Haha it’s about time you get a shower you stinky cat!” said one of the boys.
The red headed boy saw how sad the cat was.  He walked over to it.
“Don’t think I’ll let you play basketball with us again if you touch that dirty cat”, said one of the boys.
The red headed boy bent down, and pet the cat anyways.
He said to the cat, “hey there real friend.  My name is Bruno, and I’m going to take you to your new home, my home”.


"""

lying_text = """
Two girls, Sarah and Rachel, have been best friends since they were toddlers.
They are now in middle school, and are taking the same math class together.

•“Hey Sarah, have you studied for that math test yet?” asked Rachel.
“Yeah.  I’ve studied a lot for it.  What about you?” asked Sarah.

“I haven’t studied at all.  But I feel confident that I will do well anyways”, said Rachel.

It is the day of the test now.  They are sitting at their desks.
“Alright everyone, I am about to pass your tests out.  Remember, no cheating.
You have 40 minutes to finish”, says the teacher.

Sarah says to herself,  “Hmm I better make sure I read each question carefully, so that I don’t miss anything, and answer any questions wrong”.

Sarah looks over at her best friend Rachel.  She sees Rachel bubbling in every answer quickly on her answer sheet. Sarah says to herself, “But how could she know the answer to each question so quickly?  She didn’t even study.  She’s just probably guessing then.”

Then Sarah sees Rachel looking over the shoulder of the kid who is sitting right in front of her. It looks like she is copying his answer sheet.

Rachel notices that Sarah is looking at her.  Rachel whispers to Sarah, “Sarah shhhhh.  Don’t tell on me.  I’m your best friend”.

“Is something going on Sarah?” asks the teacher.  “Nothing is going on Ms. Craig,” answers Sarah.  Sarah continues taking the test without looking up again.


"""

stealing_text = """
Billy loved bikes.  He was riding his bike to the beach one day, when he came across the most beautiful bike he had ever seen.  He said to the boy that was riding the beautiful bike, “wow, that’s such a beautiful bike!”  “Thanks!” said the boy.

“Hey, how much did you pay for that bike?” asked Billy.  “It was $300”, answered the boy.

Billy rode his bike home.  He thought to himself, “hmmm, maybe my parents will buy me that bike for my birthday”.

Billy got home. He says to his parents,“hey mom and dad.  For my birthday I would like a new bike pretty please!”.  “No Billy.  You already have a nice bike and you don’t need a new expensive one”, said his father.

Billy felt sad, and mad.  So he took his bike, and rode it to an ice cream shop.

Billy walked out of the ice cream shop with ice cream, and found a table to sit outside. “Ice cream, you always make me feel better” Billy said to himself.  Next to him was another table, where a lady was sitting.

As Billy was eating his ice cream, the lady left and forgot her wallet.  Billy went and grabbed her wallet.  He opened her wallet.  “Wow, is this really $300? With that money, I could buy that red bike!”  Billy said.

The lady that was sitting at the table next to him came back, looking for her wallet.  She said, “hello there, have you seen my wallet?  I can’t find it anywhere.”  “No, I don’t know where it is”, said Billy.


"""

# Questions (same for both comic and text formats)
cat_questions = """
1.Why did the red headed boy keep the cat?
2.What made you think of that answer?
3.What lesson could someone learn from this story?
4.What made you think of that answer?

"""

lying_questions = """
1.Why doesn’t Sarah tell the teacher that Rachel is looking at another  student’s answers?
2.What made you think of that answer?
3.What lesson could someone learn from this story?
4.What made you think of that answer?
"""

stealing_questions = """
1.Why did Billy keep the lady’s wallet?
2.What made you think of that answer?
3.What lesson could someone learn from this story?
4.What made you think of that answer?
"""

# Comic image paths (Gemini can read images directly)
cat_comic_img = "/content/drive/MyDrive/Cat.pdf"
lying_comic_img = "/content/drive/MyDrive/Lying.pdf"
stealing_comic_img = "/content/drive/MyDrive/Stealing.pdf"

## Section 4: Response Format
Specifies the exact JSON schema the model must return.
One key per item, mapped to scenario, format, and question number.

In [ ]:
# ── Section 4: Response Format ────────────────────────────────────────────
response_format = """
==================== RESPONSE FORMAT ====================
Return a single valid JSON object with exactly these 24 keys.
Each value should be a short open-ended text response as the student would write it.
Do not include any explanation or text outside the JSON.

{
    "cat_comic_1": "...",
    "cat_comic_2": "...",
    "cat_comic_3": "...",
    "cat_comic_4": "...",
    "cat_text_1": "...",
    "cat_text_2": "...",
    "cat_text_3": "...",
    "cat_text_4": "...",
    "lying_comic_1": "...",
    "lying_comic_2": "...",
    "lying_comic_3": "...",
    "lying_comic_4": "...",
    "lying_text_1": "...",
    "lying_text_2": "...",
    "lying_text_3": "...",
    "lying_text_4": "...",
    "stealing_comic_1": "...",
    "stealing_comic_2": "...",
    "stealing_comic_3": "...",
    "stealing_comic_4": "...",
    "stealing_text_1": "...",
    "stealing_text_2": "...",
    "stealing_text_3": "...",
    "stealing_text_4": "..."
}
"""

## Section 5: Instructions
Behavioral constraints to ensure realistic, coherent student response generation.

In [ ]:
# ── Section 5: Instructions ───────────────────────────────────────────────
instructions = """
==================== INSTRUCTIONS ====================
1. You MUST answer every single item. Do not leave any key empty.
2. Write exactly as a student at the specified IIR level would write.
3. Keep responses short and natural — like a real student, not an essay.
4. Do not use perfect grammar or overly sophisticated vocabulary.
5. The meta-reasoning answers (Q3, Q4) should reference BOTH the story
   AND personal experience depending on the IIR level.
6. Do not explain your reasoning or mention the IIR level anywhere.
7. Output valid JSON only — no extra text, no markdown, no code fences.
"""

## Build Prompt (Final prompt fed into Gemini)
Assembles all 5 sections into one complete prompt for the API call.
Selects the correct story and questions based on scenario and format.

In [ ]:
from vertexai.generative_models import Part

def build_prompt(level, format_type):

    # Build a dynamic JSON response format for all 12 keys of the chosen format
    dynamic_response_format = f"""
==================== RESPONSE FORMAT ====================
Return a single valid JSON object with exactly these 12 keys.
Each value should be a short open-ended text response as the student would write it.
Do not include any explanation or text outside the JSON.

{{
    "cat_{format_type}_1": "...",
    "cat_{format_type}_2": "...",
    "cat_{format_type}_3": "...",
    "cat_{format_type}_4": "...",
    "lying_{format_type}_1": "...",
    "lying_{format_type}_2": "...",
    "lying_{format_type}_3": "...",
    "lying_{format_type}_4": "...",
    "stealing_{format_type}_1": "...",
    "stealing_{format_type}_2": "...",
    "stealing_{format_type}_3": "...",
    "stealing_{format_type}_4": "..."
}}
"""

    if format_type == "text":
        prompt_text = (
            build_profile_prompt(level) + "\n\n" +
            "Below are 3 different scenarios. Please read the story and answer the questions for each.\n\n" +

            "==================== SCENARIO 1: CAT ====================\n" +
            "STORY:\n" + cat_text + "\n\n" +
            "QUESTIONS:\n" + cat_questions + "\n\n" +

            "==================== SCENARIO 2: LYING ====================\n" +
            "STORY:\n" + lying_text + "\n\n" +
            "QUESTIONS:\n" + lying_questions + "\n\n" +

            "==================== SCENARIO 3: STEALING ====================\n" +
            "STORY:\n" + stealing_text + "\n\n" +
            "QUESTIONS:\n" + stealing_questions + "\n\n" +

            dynamic_response_format + "\n\n" +
            instructions
        )
        prompt_user = prompt_text

    elif format_type == "comic":
        # Load all three comic PDFs
        with open(cat_comic_img, "rb") as f:
            cat_data = f.read()
        with open(lying_comic_img, "rb") as f:
            lying_data = f.read()
        with open(stealing_comic_img, "rb") as f:
            stealing_data = f.read()

        # Create Vertex AI Part objects for all three
        cat_part = Part.from_data(data=cat_data, mime_type="application/pdf")
        lying_part = Part.from_data(data=lying_data, mime_type="application/pdf")
        stealing_part = Part.from_data(data=stealing_data, mime_type="application/pdf")

        prompt_text = (
            build_profile_prompt(level) + "\n\n" +
            "Below are 3 comic stories attached as documents in order: Cat, Lying, and Stealing. Please read the comics and answer the corresponding questions for each.\n\n" +

            "==================== SCENARIO 1: CAT QUESTIONS ====================\n" +
            cat_questions + "\n\n" +

            "==================== SCENARIO 2: LYING QUESTIONS ====================\n" +
            lying_questions + "\n\n" +

            "==================== SCENARIO 3: STEALING QUESTIONS ====================\n" +
            stealing_questions + "\n\n" +

            dynamic_response_format + "\n\n" +
            instructions
        )

        # Combine all documents and the text into a single list payload
        prompt_user = [cat_part, lying_part, stealing_part, prompt_text]

    return System_Prompt, prompt_user

In [ ]:
#API call
import vertexai
from vertexai.generative_models import GenerativeModel, Part
import time
from google.colab import auth
auth.authenticate_user()

# initialize Vertex AI with global location
PROJECT_ID = "your-gcp-project-id"  # replace with your Google Cloud project ID
vertexai.init(project=PROJECT_ID, location="global")

def call_api(prompt_system, prompt_user, max_retries=3):
    model = GenerativeModel(
        model_name="gemini-3-flash-preview",
        system_instruction=prompt_system
    )

    for attempt in range(max_retries):
        try:
            # prompt_user will now dynamically pass either the text string or the [Part, text] list
            response = model.generate_content(
                prompt_user,
                generation_config={
                    "temperature": 1.0,
                    "response_mime_type": "application/json"
                }
            )
            return response.text

        except Exception as e:
            print(f"  [Attempt {attempt+1}] Error: {e}")
            time.sleep(2 ** attempt)

    print(f"API call failed after {max_retries} retries")
    return None

In [ ]:
call_api("You are a teacher who studies student performance in comics and text reading and comprehension.", "Answer the question, what is the moral of the story of 'Goldilocks and the Three Bears'")

In [ ]:
# Response Parsing (produce python dictionary (eg. {cat_comic_1: AI answer1} ))
ITEM_KEYS = [
    "cat_comic_1", "cat_comic_2", "cat_comic_3", "cat_comic_4",
    "cat_text_1",  "cat_text_2",  "cat_text_3",  "cat_text_4",
    "stealing_comic_1", "stealing_comic_2", "stealing_comic_3", "stealing_comic_4",
    "stealing_text_1",  "stealing_text_2",  "stealing_text_3",  "stealing_text_4",
    "lying_comic_1", "lying_comic_2", "lying_comic_3", "lying_comic_4",
    "lying_text_1",  "lying_text_2",  "lying_text_3",  "lying_text_4",
]

def parse_response(json_str):
    import json

    # start with all keys as None
    result = {key: None for key in ITEM_KEYS}

    if json_str is None:
        return result

    try:
        parsed = json.loads(json_str)
    except Exception as e:
        print(f"JSON parse error: {e}")
        return result

    for key in ITEM_KEYS:
        val = parsed.get(key)
        if val is not None:
            result[key] = str(val)

    return result

In [ ]:
# Scoring
# All 24 items are open-ended and require human rating/autograder
# Scores: 0 = no response, 1 = text-implicit,
#         2 = script-implicit, 3 = combination


In [ ]:
import json
import os
import time
import pandas as pd
import random

OUTPUT_DIR = "IIR_outputs"
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FORMATS = ["comic", "text"]

def run_pipeline(total_n=100, seed=42, resume=True):
    random.seed(seed)

    # 1. Check for existing progress
    start_i = 0
    resumed = False
    raw_json = [None] * total_n
    raw_answers = [None] * total_n
    level_assignments = generate_sample(total_n)

    # --- PER-LEVEL EXACT RATIO LOGIC ---
    level_counts = {}
    for lvl in level_assignments:
        level_counts[lvl] = level_counts.get(lvl, 0) + 1

    format_pools = {}
    for lvl, count in level_counts.items():
        # count // 2 guarantees a perfect 50/50 split for even numbers.
        num_comic = count // 2
        num_text = count - num_comic

        pool = ["comic"] * num_comic + ["text"] * num_text
        random.shuffle(pool)
        format_pools[lvl] = pool

    format_assignments = []
    for lvl in level_assignments:
        format_assignments.append(format_pools[lvl].pop())
    # -----------------------------------

    if resume and os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            prog = json.load(f)

        # Added "exact_ratio" tag to force a fresh start and ignore old progress files
        if prog.get("total_n") == total_n and prog.get("seed") == seed and prog.get("exact_ratio") == True:
            level_assignments = prog["level_assignments"]
            format_assignments = prog["format_assignments"]
            raw_json = prog["raw_json"]
            raw_answers = prog["raw_answers"]
            start_i = prog["completed"]
            resumed = True
            print(f"==> Resuming from progress file: {start_i}/{total_n} already completed.")
        else:
            print("==> Progress file found but parameters differ or are from old version. Starting fresh.")

    # 2. Generate respondent IDs
    resp_ids = [f"R{str(i+1).zfill(4)}" for i in range(total_n)]

    print(f"Level distribution: {pd.Series(level_assignments).value_counts().to_dict()}")

    # Print a breakdown to verify the ratio within each level
    df_verify = pd.DataFrame({"Level": level_assignments, "Format": format_assignments})
    print("Format breakdown per level:")
    print(df_verify.groupby(['Level', 'Format']).size().unstack(fill_value=0).to_dict('index'))

    # 3. API calls
    if start_i >= total_n:
        print("All responses already generated. Skipping API calls.")
    else:
        for i in range(start_i, total_n):
            level = level_assignments[i]
            fmt = format_assignments[i]
            print(f"\r[{i+1}/{total_n}] Level: {level} | Format: {fmt}", end="")

            prompt_system, prompt_user = build_prompt(level, fmt)

            raw_json[i] = call_api(prompt_system, prompt_user)
            raw_answers[i] = parse_response(raw_json[i])

            with open(PROGRESS_FILE, "w") as f:
                json.dump({
                    "total_n": total_n,
                    "seed": seed,
                    "exact_ratio": True,  # Tag this file so it knows it uses the new balanced logic
                    "level_assignments": level_assignments,
                    "format_assignments": format_assignments,
                    "raw_json": raw_json,
                    "raw_answers": raw_answers,
                    "completed": i + 1
                }, f)

            time.sleep(0.5)

        print("\nAPI calls complete.")

    # 4. Build answer dataframe
    answer_df = pd.DataFrame({
        "respondent_id": resp_ids,
        "profile_level": level_assignments,
        "format": format_assignments
    })

    for key in ITEM_KEYS:
        answer_df[key] = [
            ans[key] if ans and key in ans else None
            for ans in raw_answers
        ]

    # 5. Save outputs
    answer_df.to_csv(os.path.join(OUTPUT_DIR, "IIR_raw_answers_V2.csv"), index=False)
    with open(os.path.join(OUTPUT_DIR, "IIR_raw_json.json"), "w") as f:
        json.dump(raw_json, f)

    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)

    print(f"Saved to: {OUTPUT_DIR}")
    print("  - IIR_raw_answers_V2.csv  (all responses)")
    print("  - IIR_raw_json.json    (raw API responses)")

    return answer_df

In [ ]:
# ── Run ────────────────────────────────────────────────────────────────────
print(f"*** Output: {OUTPUT_DIR} ***")
results = run_pipeline(total_n=16, seed=42, resume=False)


In [ ]:
import pandas as pd

# Set pandas to show all columns
pd.set_option('display.max_columns', None)

# Now when you call your dataframe, all columns will be visible


df = pd.read_csv("IIR_outputs/IIR_raw_answers_V2.csv")
df

In [ ]:
import shutil

# copy to Google Drive so you can open as Google Sheets
shutil.copy(
    "IIR_outputs/IIR_raw_answers_V2.csv",
    "/content/drive/MyDrive/IIR_raw_answers_V2.csv"
)
print("✅ Saved to Google Drive!")